# 07 — Online Feature Store Serving

This notebook demonstrates **low-latency feature retrieval** from the Snowflake Online Feature Store. The online store is backed by a Hybrid Table that mirrors the managed Dynamic Table, delivering single-digit millisecond lookups for real-time inference.

**Prerequisites:** Run `02_feature_engineering.ipynb` first to register the feature view and enable online serving.

| Concept | Details |
|---|---|
| **Offline store** | Dynamic Table — batch queries, training data generation |
| **Online store** | Hybrid Table — key-value lookups, real-time inference |
| **Target lag** | 1 minute (online mirrors offline within this window) |
| **Join keys** | `SKU_ID`, `STORE_ID` |

In [ ]:
# Session setup
import sys, os
work_dir = os.getcwd()
sys.path.insert(0, os.path.join(work_dir, 'source'))

from snowflake.snowpark.context import get_active_session
import config as cfg
import pandas as pd
import numpy as np
import time

session = get_active_session()
session.sql(f"USE DATABASE {cfg.SNOWHEALTH_DATABASE}").collect()
session.sql(f"USE SCHEMA {cfg.SNOWHEALTH_FEATURES_SCHEMA}").collect()
print(f"Session ready: {cfg.SNOWHEALTH_DATABASE}.{cfg.SNOWHEALTH_FEATURES_SCHEMA}")

In [ ]:
# Connect to the Feature Store and retrieve the registered feature view
from snowflake.ml.feature_store import FeatureStore, CreationMode
from snowflake.ml.feature_store.feature_view import StoreType

fs = FeatureStore(
    session=session,
    database=cfg.SNOWHEALTH_DATABASE,
    name=cfg.SNOWHEALTH_FEATURES_SCHEMA,
    default_warehouse="COMPUTE_WH",
    creation_mode=CreationMode.FAIL_IF_NOT_EXIST,
)

fv = fs.get_feature_view(cfg.SNOWHEALTH_FEATURE_VIEW_NAME, cfg.SNOWHEALTH_FEATURE_VIEW_VERSION)
print(f"Feature View: {fv.name} v{fv.version}")
print(f"Online enabled: {fv.online}")
print(f"Features ({len(fv.feature_names)}): {fv.feature_names[:8]}...")
print(f"Entity join keys: SKU_ID, STORE_ID")

In [ ]:
# Single-key online lookup — the core real-time pattern
# This is what a production inference service would call
start = time.time()
result = fs.read_feature_view(
    cfg.SNOWHEALTH_FEATURE_VIEW_NAME,
    cfg.SNOWHEALTH_FEATURE_VIEW_VERSION,
    keys=[["RX-001", "SH-1001"]],
    store_type=StoreType.ONLINE,
)
online_row = result.to_pandas()
elapsed_ms = (time.time() - start) * 1000

print(f"Single-key lookup: {elapsed_ms:.0f} ms")
print(f"Retrieved {len(online_row)} row, {len(online_row.columns)} features")
print(f"\nSample features for SKU=RX-001, Store=SH-1001:")
key_features = ['UNITS_SOLD', 'ROLLING_7D_AVG_UNITS', 'ROLLING_30D_AVG_UNITS',
                'SEASONAL_DEMAND_INDEX', 'INVENTORY_TURNOVER_RATIO', 'TEMPERATURE_F']
for feat in key_features:
    if feat in online_row.columns:
        print(f"  {feat}: {online_row[feat].iloc[0]}")

In [ ]:
# Multi-key batch lookup — fetch features for multiple SKU-Store pairs at once
keys_to_lookup = [
    ["RX-001", "SH-1001"],
    ["RX-002", "SH-1002"],
    ["RX-003", "SH-1003"],
    ["OTC-001", "SH-1001"],
    ["OTC-002", "SH-1005"],
]

start = time.time()
result = fs.read_feature_view(
    cfg.SNOWHEALTH_FEATURE_VIEW_NAME,
    cfg.SNOWHEALTH_FEATURE_VIEW_VERSION,
    keys=keys_to_lookup,
    store_type=StoreType.ONLINE,
)
batch_df = result.to_pandas()
elapsed_ms = (time.time() - start) * 1000

print(f"Multi-key lookup ({len(keys_to_lookup)} keys): {elapsed_ms:.0f} ms")
print(f"Retrieved {len(batch_df)} rows\n")
print(batch_df[['SKU_ID', 'STORE_ID', 'UNITS_SOLD', 'ROLLING_7D_AVG_UNITS',
                'SEASONAL_DEMAND_INDEX']].to_string(index=False))

In [ ]:
# Selective feature retrieval — only fetch the features you need
# Reduces payload size and latency for production serving
selected_features = [
    'UNITS_SOLD', 'ROLLING_7D_AVG_UNITS', 'ROLLING_30D_AVG_UNITS',
    'SEASONAL_DEMAND_INDEX', 'INVENTORY_TURNOVER_RATIO',
    'PRICE_TIMES_PROMO', 'TEMPERATURE_F', 'IS_PROMOTION_INT',
]

start = time.time()
result = fs.read_feature_view(
    cfg.SNOWHEALTH_FEATURE_VIEW_NAME,
    cfg.SNOWHEALTH_FEATURE_VIEW_VERSION,
    keys=[["RX-001", "SH-1001"]],
    feature_names=selected_features,
    store_type=StoreType.ONLINE,
)
slim_df = result.to_pandas()
elapsed_ms = (time.time() - start) * 1000

print(f"Selective lookup ({len(selected_features)} features): {elapsed_ms:.0f} ms")
print(f"Columns returned: {list(slim_df.columns)}")
print(f"\nValues:")
for col in slim_df.columns:
    if col not in ['SKU_ID', 'STORE_ID']:
        print(f"  {col}: {slim_df[col].iloc[0]}")